# QuantumFlow: calling the C++ engine from Python
This notebook uses the `qf_core` module built by CMake (pybind11). Run it from the repository root after `./run_all.sh build`.

In [ ]:
import sys, time, numpy as np
sys.path.insert(0, 'python')
import qf_core
print('CUDA available:', qf_core.cuda_available(), qf_core.cuda_device_name())

## 1. One option, three ways
The analytic Heston price is the reference; Monte Carlo should land within a few standard errors of it.

In [ ]:
p = qf_core.HestonParams(S0=100, K=100, T=1.0, r=0.03, v0=0.04, kappa=2.0, theta=0.04, xi=0.5, rho=-0.7)
print('analytic   ', qf_core.heston_price(p))
print('MC (auto)  ', qf_core.mc_price(p, n_paths=1<<20, n_steps=256))
print('Black-Scholes at 20% vol', qf_core.bs_call(100, 100, 1.0, 0.03, 0.2))

## 2. Zero-copy batches
`heston_greeks_batch` reads the NumPy array in place (buffer protocol) and returns C++-owned memory wrapped as a NumPy array (no copy). `torch.from_numpy` then shares that same memory.

In [ ]:
X = np.tile([100,100,1,0.03,0.04,2,0.04,0.5,-0.7], (1000,1)).astype(np.float64)
X[:,1] = np.linspace(70, 130, 1000)   # sweep the strike
t = time.perf_counter(); G = qf_core.heston_greeks_batch(X); print(f'{(time.perf_counter()-t)*1e3:.1f} ms for 1000 options')
print('owns its data?', G.flags['OWNDATA'], '(False = memory belongs to C++)')
try:
    import torch; T = torch.from_numpy(G); print('torch shares memory:', T.data_ptr() == G.ctypes.data)
except ImportError: pass

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(X[:,1], G[:,1]); ax[0].set_title('Delta vs strike'); ax[0].set_xlabel('K')
ax[1].plot(X[:,1], G[:,2]); ax[1].set_title('Gamma vs strike'); ax[1].set_xlabel('K')
plt.show()

## 3. The order book from Python

In [ ]:
book = qf_core.OrderBook(1 << 14, 0.01)
mids, ns = book.run_feed(1_000_000, 7, 10_000)
print(f'{ns/1e6:.1f} ms for 1M messages = {ns/1e6:.0f} ns/msg (includes mid sampling)')
print('top of book:', book.best_bid(), book.best_ask(), 'live orders', book.live_orders())
plt.plot(mids); plt.title('mid price over the feed'); plt.show()